# Урок 5.6. Решение типичных проблем в продакшене

Модуль 5 «Локальные LLM: запуск без облака и без GPU» · урок 6 из 7

Урок 5.5 научил замечать, что что-то не так. Этот — разбираться, **что именно**
и как чинить. Формат — врачебный справочник: пять болезней локальной LLM, у каждой
симптомы → диагностика → лечение. Все болезни мы вызовем у себя сами, на живой
модели, — контролируемая авария лучше внезапной.

## 🎯 Что вы узнаете

- Память: откуда берётся OOM, как num_ctx раздувает KV-кеш (замерим) и почему swap — не спасение
- Скорость: лесенка частичной выгрузки на GPU и чек-лист «почему вдруг медленно»
- Зависшие запросы: чем опасен запрос без num_predict и отменяет ли Ollama брошенную генерацию (проверим!)
- Конкуренция: как выглядит очередь при NUM_PARALLEL=1 глазами трёх клиентов
- Версии: почему тег модели — не версия, и как жить с digest и канарейкой

## Карта болезней

| Симптом | Болезнь | Раздел |
|---|---|---|
| 500 «requires more memory», OOM-killer, хост свопит | память | 1 |
| «вчера летало, сегодня ползёт» | скорость | 2 |
| запрос висит минутами, клиенты копятся | зависания | 3 |
| у всех медленно, когда пользователей больше одного | конкуренция | 4 |
| «после обновления модель поглупела» | версии | 5 |


## Болезнь 1. Память: OOM и его тихий брат swap

**Симптомы.** Ollama отвечает 500 с текстом вроде `model requires more system
memory`; на Linux OOM-killer убивает процесс (в `dmesg` — `Out of memory: Killed`);
на Windows система не убивает, а **свопит** — всё «работает», но в 10–100 раз
медленнее (это тихий брат OOM, и он коварнее).

**Диагностика.** Арифметика из урока 5.2: `файл модели + KV-кеш + runtime` против
доступной RAM/VRAM (`psutil`, `/api/ps`). Про файл и runtime мы всё знаем;
недооценивают обычно **KV-кеш**, потому что он зависит не от модели, а от вашего
`num_ctx` — и от `NUM_PARALLEL`, который его умножает (урок 5.3).

Проверим на живой модели: загрузим её с разным `num_ctx` и спросим у `/api/ps`,
сколько памяти запланировал демон:


In [ ]:
import time

import httpx

OLLAMA_URL = "http://127.0.0.1:11434"  # не localhost — урок 5.3
NS = 1_000_000_000
GB = 1024**3
MODEL = "qwen2.5:3b"


def ollama_available() -> bool:
    try:
        return httpx.get(f"{OLLAMA_URL}/api/version", timeout=2).status_code == 200
    except httpx.HTTPError:
        return False


def unload_model(model: str = MODEL) -> None:
    httpx.post(f"{OLLAMA_URL}/api/generate", json={"model": model, "keep_alive": 0}, timeout=60)
    time.sleep(1)


def ps_info() -> tuple[float, float] | None:
    """(память в ГБ, доля на GPU) для загруженной модели, либо None."""
    models = httpx.get(f"{OLLAMA_URL}/api/ps", timeout=5).json()["models"]
    if not models:
        return None
    m = models[0]
    return m["size"] / GB, m.get("size_vram", 0) / m["size"]


OLLAMA_UP = ollama_available()
if OLLAMA_UP:
    print(f"{'num_ctx':>8} | {'память, ГБ':>10}")
    print("-" * 22)
    for ctx in (2048, 4096, 16384, 32768):
        unload_model()
        httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL, "prompt": "ping", "stream": False,
            "options": {"num_predict": 1, "num_ctx": ctx},
        }, timeout=300).raise_for_status()
        info = ps_info()
        if info is None:  # загрузка сорвалась — см. server.log, msg="Load failed"
            print(f"{ctx:>8} | модель не загрузилась, повторите ячейку")
            continue
        size, _ = info
        print(f"{ctx:>8} | {size:>10.2f}")
else:
    print("Ollama не запущена — живые ячейки будут пропущены")

У меня: 1.94 ГБ на контексте 2k и **3.13 ГБ на 32k** — тот же файл, а памяти
на +1.2 ГБ больше. Это и есть KV-кеш: ~39 КБ на токен контекста у этой модели
(3B с GQA). Теперь умножьте на `NUM_PARALLEL` — каждый слот получает свой контекст —
и «почему 7B не влезла в 16 ГБ» перестаёт быть загадкой.

**Лечение** (по возрастанию цены):

| Мера | Что даёт | Урок |
|---|---|---|
| Ограничить `num_ctx` под реальную задачу | −сотни МБ, быстрее prefill | этот |
| `OLLAMA_MAX_LOADED_MODELS=1` | вторая модель не съест RAM | 5.3 |
| Квантизация агрессивнее (Q4 вместо Q8) | файл и трафик RAM вдвое меньше | 5.2 |
| Модель меньше (3B вместо 7B) | −40–60% всего | 5.2 |
| Потолок памяти: `MemoryMax` / docker limits | падение вместо swap-агонии | 5.3 |

Последняя строка не опечатка: **упасть лучше, чем свопить**. Упавший сервис
перезапустит супервизор, и алерт OllamaDown сработает за минуту (урок 5.5).
Свопящий сервис «работает» часами, отравляя всё вокруг, — и алерта «просто всё
плохо» не существует. Именно поэтому в `docker-compose.yml` модуля стоит
жёсткий `memory: 12g`, а swap мы мониторим отдельным правилом.


## Болезнь 2. «Вчера летало, сегодня ползёт»

Чек-лист причин — в порядке дешевизны проверки (первые две мы уже видели
в уроке 5.5):

1. **Модель съехала с GPU — полностью или частично.** VRAM заняли браузер, игра,
   вторая модель. Диагноз: `size_vram` в `/api/ps` (или `ollama ps` — колонка
   PROCESSOR вида `45%/55% CPU/GPU`).
2. **Хост в swap** — `psutil.swap_memory()`.
3. **Троттлинг / энергоплан** — ноутбук на батарее режет частоты (мы видели ±20%
   даже в замерах урока 5.4).
4. **Фоновая нагрузка на CPU** — индексатор, антивирус, соседний процесс.
5. **Контекст раздулся** — prefill на CPU идёт ~50–60 ток/с (урок 5.2): промпт
   на 2000 токенов — это ~35 секунд *до первого токена ответа*.
6. **Сломался кеш префикса** — изменили system-промпт, и каждый запрос платит
   полный prefill вместо кешированного (тот же урок 5.2).

Причину №1 стоит увидеть глазами — она самая частая на машинах с GPU и самая
«ступенчатая». Ollama умеет класть на GPU **часть слоёв** (`num_gpu` — число слоёв):
именно это она делает сама, когда VRAM мало. Пройдём по лесенке:


In [ ]:
if OLLAMA_UP:
    print(f"{'num_gpu':>7} | {'ток/с':>6} | {'доля на GPU':>11}")
    print("-" * 32)
    for num_gpu in (999, 18, 8, 0):  # 999 = все слои; у qwen2.5:3b их 36
        unload_model()
        d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL,
            "prompt": f"Вопрос {num_gpu}: назови 12 животных через запятую.",  # уникальные
            "stream": False,
            "options": {"temperature": 0, "seed": 42, "num_predict": 40, "num_gpu": num_gpu},
        }, timeout=600).raise_for_status().json()
        tps = d["eval_count"] / (d["eval_duration"] / NS)
        _, share = ps_info()
        print(f"{num_gpu:>7} | {tps:>6.1f} | {share:>11.0%}")
    unload_model()  # вернуть дефолт (опции залипают с runner'ом — уроки 5.2, 5.3)
else:
    print("Пропущено: Ollama не запущена")

Моя лесенка (в разных прогонах числа гуляют на ±20% — троттлинг): **62–68 ток/с
на полном GPU → 25–34 на половине слоёв → 17–25 на трети → 13–18 на чистом CPU**. Обратите внимание: даже 50% слоёв на GPU — это уже двукратная
потеря, потому что каждый токен проходит через ВСЕ слои, и CPU-половина становится
бутылочным горлышком.

В жизни `num_gpu` за вас «крутит» сама Ollama при нехватке VRAM — молча. Поэтому
детектор из урока 5.5 (доля VRAM < 90% дольше двух проверок) — обязательный
алерт для GPU-машин. Лечение: освободить VRAM (закрыть тяжёлые приложения,
выгрузить вторую модель), уменьшить `num_ctx` (KV-кеш тоже живёт в VRAM —
см. болезнь 1) или честно принять CPU-скорость и поднять таймауты.


## Болезнь 3. Зависшие запросы

**Как возникают.** Три типовых источника:

- **Запрос без `num_predict`**: модель имеет право писать до конца контекста.
  4000 токенов на CPU — это четыре с лишним минуты; клиент давно ушёл, а слот занят.
- **Забитая очередь** (урок 5.3): запрос стоит за чужими минутами.
- **Больная загрузка**: модель грузится дольше `OLLAMA_LOAD_TIMEOUT` — умирающий
  диск, память под давлением, капризы GPU-драйвера. Мы поймали такое живьём при
  подготовке урока: одна загрузка зависла и упала через 13 минут
  (`msg="Load failed" ... timed out` в server.log), после чего демон сам пришёл
  в норму. Диагноз — всегда в логе демона (урок 5.3), лечение — повторная
  загрузка, а при рецидивах — рестарт службы и проверка диска/драйвера.

**Оборона** — с двух сторон. Со стороны запроса: `num_predict` ставим **всегда**
(это потолок расходов, как лимит в SQL-запросе). Со стороны клиента: раздельные
таймауты connect/read (урок 5.3).

Остаётся вопрос на миллион: клиент отвалился по таймауту — **а демон продолжает
считать брошенный запрос?** Если да, то толпа нетерпеливых клиентов с retry
устроит снежный ком. Проверим экспериментом: запустим потенциально бесконечную
генерацию с коротким клиентским таймаутом и сразу после обрыва замерим,
как быстро ответит следующий запрос:


In [ ]:
if OLLAMA_UP:
    # прогреваем модель с дефолтными опциями
    httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": MODEL, "prompt": "ping", "stream": False,
        "options": {"num_predict": 1}}, timeout=300)

    t0 = time.perf_counter()
    try:
        httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL,
            "prompt": "Выпиши все числа от 1 до 2000 через запятую, не останавливайся.",
            "stream": False,
            "options": {"temperature": 0},  # num_predict НЕ задан — намеренно!
        }, timeout=4.0)
        print("не должно было успеть!")
    except httpx.ReadTimeout:
        print(f"клиент отвалился по ReadTimeout через {time.perf_counter() - t0:.1f} с")

    t0 = time.perf_counter()
    httpx.post(f"{OLLAMA_URL}/api/generate", json={
        "model": MODEL, "prompt": "Скажи слово «эхо».", "stream": False,
        "options": {"num_predict": 5},
    }, timeout=300).raise_for_status()
    print(f"следующий запрос ответил за {time.perf_counter() - t0:.1f} с")
else:
    print("Пропущено: Ollama не запущена")

У меня следующий запрос ответил за ~1–1.5 секунды — значит, **Ollama отменила
брошенную генерацию, когда клиент разорвал соединение**. Хорошая новость, но
с двумя оговорками:

- Это поведение конкретной версии демона — не полагайтесь на него вслепую,
  наш тест занимает три строки, прогоните его на своей (весь смысл урока:
  не верить, а мерить).
- Отмена срабатывает при **разрыве соединения**. Если ваш клиент таймаутит,
  но соединение не закрывает (пулы, прокси, буферизация), запрос продолжит
  считаться. Ещё одна причина использовать нормальный HTTP-клиент с явными
  таймаутами (наш `RobustOllamaClient` из урока 5.3).

И повторим мантру: `ReadTimeout` при генерации — **не** повод для retry
(матрица урока 5.3): повтор удвоит нагрузку, которая и так не справилась.


## Болезнь 4. Конкурентный доступ

При `OLLAMA_NUM_PARALLEL=1` (наш дефолт из урока 5.3) демон обрабатывает один
запрос, остальные ждут в очереди. Клиенту очередь не видна: у него просто
«медленно». Посмотрим на неё глазами трёх одновременных клиентов:


In [ ]:
import threading

if OLLAMA_UP:
    results = []

    def worker(client_id: int) -> None:
        t0 = time.perf_counter()
        d = httpx.post(f"{OLLAMA_URL}/api/generate", json={
            "model": MODEL,
            "prompt": f"Клиент {client_id}: назови 15 стран через запятую.",
            "stream": False,
            "options": {"temperature": 0, "seed": 42, "num_predict": 60},
        }, timeout=600).raise_for_status().json()
        results.append((client_id, time.perf_counter() - t0, d["eval_duration"] / NS))

    threads = [threading.Thread(target=worker, args=(i,)) for i in range(3)]
    for t in threads:
        t.start()
    for t in threads:
        t.join()

    for client_id, wall, gen_s in sorted(results, key=lambda r: r[1]):
        print(f"клиент {client_id}: полное время {wall:4.1f} с "
              f"(чистая генерация {gen_s:.1f} с)")
else:
    print("Пропущено: Ollama не запущена")

Лесенка как по учебнику: у меня чистая генерация у всех ~1 секунда, а полное
время — **2.7 / 3.7 / 4.7 секунды**: каждый следующий в очереди ждёт всех
предыдущих. При десяти клиентах хвост очереди ждал бы десять секунд — и, скорее
всего, отвалился бы по таймауту, породив retry и снежный ком.

Что с этим делать (всё уже в нашем арсенале):

- **Поднять `NUM_PARALLEL`?** На CPU — нет: полоса памяти одна, каждому станет
  хуже, а KV-кеш умножится (урок 5.3). На GPU с запасом VRAM — можно попробовать 2–4.
- **Ограничить конкуренцию на своей стороне**: семафор перед LLM, как мы делали
  в сервисе урока 3.2 (`RAG_MAX_CONCURRENT_GEN=1`), плюс честный `queue_ms` в ответе.
- **Короткая очередь + быстрый 503 + retry с джиттером** (урок 5.3) — лучше
  честного отказа, чем бесконечного ожидания.
- **Вторая нода / меньшая модель** — если очередь хроническая, это уже не инцидент,
  а ёмкость (расчёт потолка — урок 5.1).


## Болезнь 5. Версии: «после обновления модель поглупела»

Тег `qwen2.5:3b` — **не версия, а указатель**: издатель может перезалить модель,
и завтрашний `ollama pull` притащит другие веса под тем же именем. Плюс демон
Ollama **сам обновляется** — мы пережили это в уроке 3.2, когда инсталлер убил
API посреди работы. Итого дрейфуют две вещи: веса и рантайм.

Истинное имя весов — **digest** (контент-адрес, как у Docker-образов и git-коммитов):


In [ ]:
def model_digest(model: str) -> str | None:
    tags = httpx.get(f"{OLLAMA_URL}/api/tags", timeout=5).json()["models"]
    for m in tags:
        if m["name"] == model:
            return m["digest"]
    return None


def assert_model_digest(model: str, expected: str) -> bool:
    """Проверка при старте сервиса: та ли модель, что мы тестировали?"""
    actual = model_digest(model)
    if actual == expected:
        print(f"OK: {model} — digest совпадает ({actual[:19]}…)")
        return True
    print(f"ВНИМАНИЕ: {model} изменилась!")
    print(f"  ожидали: {str(expected)[:19]}…")
    print(f"  на диске: {str(actual)[:19]}…")
    print("  прогоните бенчмарк урока 5.2 перед выкаткой!")
    return False


if OLLAMA_UP:
    pinned = model_digest(MODEL)   # это «замороженное» значение кладут в конфиг сервиса
    print(f"текущий digest {MODEL}: {pinned[:19]}…\n")
    assert_model_digest(MODEL, pinned)                 # штатный старт
    assert_model_digest(MODEL, "sha256:другие-веса")   # а так выглядит дрейф
else:
    print("Пропущено: Ollama не запущена")

Процедура безопасного обновления модели — три шага:

1. **Заморозить текущее**: `ollama cp qwen2.5:3b qwen2.5-3b-prod` — локальный тег,
   который `ollama pull` не перезапишет. Сервис ходит в `qwen2.5-3b-prod`.
2. **Обновиться рядом**: `ollama pull qwen2.5:3b` обновляет только «плавающий» тег.
3. **Канарейка**: прогнать `scripts/bench.py` (наш регрессионный тест из урока 5.2)
   на новой версии. Счёт не упал — перевесить прод-тег: `ollama cp qwen2.5:3b
   qwen2.5-3b-prod`. Упал — вы только что бесплатно избежали инцидента.

Digest при этом пишется в конфиг сервиса, а `assert_model_digest` живёт
в `/healthz` (рифма с проверкой зависимостей из урока 3.2). Версию демона
(`/api/version`) тоже логируйте при старте: «поглупела модель» иногда означает
«обновился рантайм».

## Собираем всё: diagnose.py

Когда горит алерт, некогда вспоминать команды. `scripts/diagnose.py` — одна
кнопка «снимите мне показания»: окружение, RAM/swap, демон, модели против
свободной памяти, кто в памяти и на каком проценте GPU, плюс опциональная проба
генерации с вердиктами. Запустим как его запускают в бою:


In [ ]:
import subprocess
import sys

if OLLAMA_UP:
    proc = subprocess.run(
        [sys.executable, "scripts/diagnose.py", "--probe", MODEL],
        capture_output=True, text=True, timeout=300, encoding="utf-8", errors="replace",
    )
    print(proc.stdout)
    unload_model()  # проба загрузила модель
else:
    print("Пропущено: Ollama не запущена")

Такой скрипт — первое, что просят прислать при разборе инцидента («выполните
и пришлите вывод»). Обратите внимание на встроенный детектор болезни 2: если
модель загружена на GPU частично, diagnose честно ругается WARN.

## ⚠️ Частые ошибки

1. **Лечить медленную генерацию ядрами CPU.** Она память-bound (уроки 5.2, 5.4):
   больше ядер ≠ быстрее, после 4–8 потоков бывает хуже. Помогают полоса памяти,
   GPU или модель меньше.
2. **Считать swap решением OOM.** Swap меняет громкое падение на тихую агонию
   в 10–100 раз медленнее — худший из миров для сервиса (и незаметный для L1/L2
   health-check). Лучше жёсткий потолок памяти и честный рестарт.
3. **`num_ctx=32k` «на всякий случай».** Это +1.2 ГБ памяти на нашей 3B (замер выше),
   медленнее prefill и толще KV на каждый слот параллелизма. Контекст — под задачу.
4. **Обновляться без канарейки.** И модель (тег мутабелен!), и демон (история
   с самообновлением из урока 3.2) меняются под ногами. Digest в конфиг, бенчмарк
   после pull, версия демона в лог.
5. **Перезапуск демона как универсальное лекарство.** Часто помогает — и стирает
   улики (что было в памяти, какая доля GPU, что в swap). Сначала `diagnose.py`
   снимок, потом рестарт.
6. **Запросы без `num_predict`.** Право писать до конца контекста — это минуты
   занятого слота. Потолок ставится всегда, как LIMIT в SQL.

## Упражнения

1. **Бюджет памяти.** Напишите `fits_in_ram(file_gb, num_ctx, kv_kb_per_token,
   num_parallel, available_gb)` и разберите сценарий: хост 8 ГБ, модель 7B-Q4
   (4.5 ГБ файл), `num_ctx=8192`, `NUM_PARALLEL=2`. Влезет? Кто главный виновник?
   Предложите три исправления с расчётом.
2. **safe_generate().** Соберите из уроков 5.3 и 5.6 обёртку над `/api/generate`,
   в которой невозможно забыть про безопасность: `num_predict` обязателен,
   таймауты connect/read раздельные, retry только по матрице (503/connect —
   да, ReadTimeout — нет).
3. **Канарейка после pull.** Напишите процедуру `check_model_update(model,
   pinned_digest)`: сравнить digest с законсервированным; если отличается —
   прогнать `scripts/bench.py` и вернуть вердикт «можно перевешивать прод-тег /
   нельзя». (Сам `ollama pull` в решении не выполняем — только логика вокруг.)

Решения — ниже.


## Мини-квиз

<details><summary><b>1. Ollama вернула 500 «model requires more system memory». Порядок действий?</b></summary>

Диагностика: сколько нужно (формула урока 5.2: файл + KV(num_ctx) + runtime,
умножить KV на NUM_PARALLEL) против `psutil.virtual_memory().available`.
Лечение по возрастанию цены: уменьшить `num_ctx`, `MAX_LOADED_MODELS=1`,
квантизация ниже, модель меньше. Swap решением не считается.
</details>

<details><summary><b>2. `ollama ps` показывает PROCESSOR = «45%/55% CPU/GPU». Что это и чем грозит?</b></summary>

Частичная выгрузка: в VRAM поместилась только часть слоёв, остальные на CPU.
Каждый токен проходит через все слои, поэтому скорость ближе к CPU, чем к GPU
(наша лесенка: 50% слоёв = половина скорости). Причина — VRAM занята кем-то ещё
или KV-кеш раздут большим num_ctx.
</details>

<details><summary><b>3. Чем опасен запрос без num_predict и почему это не чинится таймаутом клиента?</b></summary>

Модель вправе генерировать до конца контекста — минуты занятого слота, за которым
копится очередь. Клиентский таймаут спасает клиента, но слот освобождается только
если демон отменяет брошенный запрос при разрыве соединения — это надо проверять
на своей версии (наш тест: 3 строки). Потолок `num_predict` — обязанность запроса.
</details>

<details><summary><b>4. Три клиента одновременно, NUM_PARALLEL=1: чистая генерация у всех по 1 с, а полные времена 2.7/3.7/4.7 с. Почему и что делать?</b></summary>

Очередь сериализует: каждый ждёт предыдущих (плюс общий prefill/накладные).
Меры: семафор и honest queue_ms на своей стороне (урок 3.2), короткая очередь +
503 + retry с джиттером (урок 5.3), на GPU — попробовать NUM_PARALLEL 2–4;
на CPU параллелизм не поднимать. Хроническая очередь = вопрос ёмкости, не инцидент.
</details>

<details><summary><b>5. Почему «прибить и перезапустить демона» — плохой ПЕРВЫЙ шаг при инциденте?</b></summary>

Рестарт стирает улики: что было загружено, какая доля VRAM, что в swap, что
в очереди. Сначала снимок (`diagnose.py`, `/api/ps`, psutil), потом рестарт.
Иначе инцидент вернётся, а данных для разбора снова не будет.
</details>

<details><summary><b>6. Чем digest модели лучше тега для продакшена?</b></summary>

Тег мутабелен: издатель перезальёт веса — и `qwen2.5:3b` завтра означает другую
модель. Digest — контент-адрес: совпал — значит, байты те же, что вы тестировали.
Процедура: digest в конфиг, `assert_model_digest` в healthz, обновление через
`ollama cp` + канареечный бенчмарк.
</details>


## Итоги

- Память: главный недооценённый едок — KV-кеш (наш замер: +1.2 ГБ от num_ctx 2k → 32k),
  умножается на NUM_PARALLEL. Упасть лучше, чем свопить: ставьте потолки памяти.
- Скорость: лесенка частичной выгрузки (у меня 62–68 → 25–34 → 17–25 → 13–18 ток/с) — самая
  частая причина «вдруг медленно» на GPU-машинах; диагноз по size_vram в /api/ps.
- Зависания: num_predict всегда; клиентские таймауты; отмену брошенных запросов
  демоном мы проверили экспериментом (отменяет при разрыве) — проверьте на своей версии.
- Конкуренция: NUM_PARALLEL=1 сериализует — полные времена растут лесенкой;
  лечится семафором/503+retry/ёмкостью, а не слепым поднятием параллелизма на CPU.
- Версии: тег — указатель, digest — истина. ollama cp для заморозки, канареечный
  бенчмарк после pull, версия демона в лог.
- `scripts/diagnose.py` — снимок всех показаний одной командой, до рестарта.

**Дальше**: урок 5.7 — финал модуля: заворачиваем всё в FastAPI-прокси
с health-check, метриками и retry.

## Полезные ссылки

- FAQ Ollama (память, VRAM, параллелизм): <https://github.com/ollama/ollama/blob/main/docs/faq.md>
- Троблшутинг Ollama: <https://github.com/ollama/ollama/blob/main/docs/troubleshooting.md>
- psutil: <https://psutil.readthedocs.io>


---

## Решения упражнений

### Упражнение 1: бюджет памяти


In [ ]:
def fits_in_ram(file_gb: float, num_ctx: int, kv_kb_per_token: float,
                num_parallel: int, available_gb: float, runtime_gb: float = 0.7):
    kv_gb = num_ctx * kv_kb_per_token / 1024 / 1024 * num_parallel
    total = file_gb + kv_gb + runtime_gb
    verdict = "влезает" if total <= available_gb else "НЕ влезает"
    print(f"файл {file_gb:.1f} + KV {kv_gb:.1f} ({num_parallel} слот(а) x {num_ctx} ток) "
          f"+ runtime {runtime_gb} = {total:.1f} ГБ из {available_gb} ГБ -> {verdict}")
    return total <= available_gb


AVAILABLE = 8.0 - 3.0  # хост 8 ГБ, ОС и фон съедают ~3

print("Сценарий из условия (7B-Q4, ctx 8k, 2 слота):")
fits_in_ram(4.5, 8192, 128, 2, AVAILABLE)

print("\nПолумера — урезать контекст и слоты, оставив 7B:")
fits_in_ram(4.5, 4096, 128, 1, AVAILABLE)

print("\nНастоящие исправления:")
fits_in_ram(3.5, 4096, 128, 1, AVAILABLE)   # 1) 7B в Q3_K_M (~3.5 ГБ файл) + скромный ctx
fits_in_ram(1.8, 8192, 128, 2, AVAILABLE)   # 2) 3B-Q4 — влезает ДАЖЕ с исходными аппетитами
fits_in_ram(1.8, 4096, 128, 1, AVAILABLE)   # 3) 3B-Q4 + скромные настройки: запас на пики

Разбор. Главный видимый виновник — KV-кеш: `8192 токенов × 128 КБ × 2 слота = 2 ГБ`.
Но обратите внимание на полумеру: даже срезав контекст и слоты, **7B-Q4 на этом
хосте не живёт** — 5.7 ГБ при 5 доступных. Честные варианты: 7B в квантизации
ниже (Q3_K_M, ~3.5 ГБ файл — и проверка качества бенчмарком урока 5.2 обязательна,
Q3 заметно глупеет) или 3B-Q4, которая влезает даже с исходными аппетитами.

Это рифма с `pick_quant` из урока 5.2, который честно возвращал `None`:
иногда правильное «лечение» памяти — признать, что модель выбрана не под этот
хост. Решает не память, а бенчмарк: если 3B проходит ваш тест — вопрос закрыт.


### Упражнение 2: safe_generate()


In [ ]:
def safe_generate(model: str, prompt: str, *, num_predict: int,
                  read_timeout_s: float = 180.0, retries: int = 2, **options) -> dict:
    """Обёртка, в которой нельзя забыть про безопасность.

    num_predict — обязательный именованный аргумент (потолок расходов);
    таймауты connect/read раздельные (урок 5.3); retry — только по матрице:
    ConnectError и 502/503 — да, ReadTimeout — никогда.
    """
    options["num_predict"] = num_predict
    timeout = httpx.Timeout(connect=5.0, read=read_timeout_s, write=10.0, pool=5.0)
    last_exc: Exception | None = None
    for attempt in range(retries + 1):
        try:
            resp = httpx.post(f"{OLLAMA_URL}/api/generate", json={
                "model": model, "prompt": prompt, "stream": False, "options": options,
            }, timeout=timeout)
            if resp.status_code in (502, 503):
                raise httpx.HTTPStatusError("busy", request=resp.request, response=resp)
            resp.raise_for_status()
            return resp.json()
        except (httpx.ConnectError, httpx.ConnectTimeout, httpx.HTTPStatusError) as exc:
            if isinstance(exc, httpx.HTTPStatusError) and exc.response.status_code not in (502, 503):
                raise                      # 4xx и прочее не ретраим
            last_exc = exc
            if attempt < retries:
                time.sleep(0.5 * 2**attempt)
        # ReadTimeout сюда не попадает намеренно: он летит наружу без повторов
    raise last_exc


if OLLAMA_UP:
    d = safe_generate(MODEL, "Скажи «готово».", num_predict=5)
    print(d["response"].strip())
    # а так — не скомпилируется даже намерение забыть потолок:
    # safe_generate(MODEL, "hi")  # TypeError: missing keyword-only 'num_predict'
    unload_model()
else:
    print("Пропущено")

### Упражнение 3: канарейка после pull


In [ ]:
import subprocess
import sys


def check_model_update(model: str, pinned_digest: str,
                       min_score: float = 0.9) -> str:
    """Вердикт после ollama pull: можно ли перевешивать прод-тег."""
    actual = model_digest(model)
    if actual is None:
        return f"СТОП: {model} не найдена на диске"
    if actual == pinned_digest:
        return "OK: digest не изменился, модель та же — делать нечего"

    print(f"digest изменился: {pinned_digest[:19]}… -> {actual[:19]}…")
    print("гоняем канареечный бенчмарк (урок 5.2)...")
    proc = subprocess.run(
        [sys.executable, "scripts/bench.py", "--models", model],
        capture_output=True, text=True, timeout=900, encoding="utf-8", errors="replace",
    )
    line = next((_l for _l in proc.stdout.splitlines() if model in _l and "%" in _l), "")
    score = float(line.split()[1].rstrip("%")) / 100 if line else 0.0
    print(f"счёт новой версии: {score:.0%} (порог {min_score:.0%})")
    if score >= min_score:
        return (f"МОЖНО: ollama cp {model} <прод-тег>; новый digest {actual[:19]}… — в конфиг")
    return "НЕЛЬЗЯ: качество просело — прод-тег не трогаем, разбираемся"


if OLLAMA_UP:
    pinned = model_digest(MODEL)
    print(check_model_update(MODEL, pinned))          # digest совпадает
    print()
    print(check_model_update(MODEL, "sha256:старый"))  # имитация дрейфа -> канарейка
    unload_model()
else:
    print("Пропущено")